In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "data" / "twitter_training.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "twitter_validation.csv"

print("Training path:", TRAIN_PATH)
print("Validation path:", VALIDATION_PATH)
print("Training file exists:", TRAIN_PATH.exists())
print("Validation file exists:", VALIDATION_PATH.exists())
COLUMN_NAMES = ["tweet_id", "entity", "sentiment", "text"]

train_df = pd.read_csv(
    TRAIN_PATH,
    header=None,
    names=COLUMN_NAMES
)

validation_df = pd.read_csv(
    VALIDATION_PATH,
    header=None,
    names=COLUMN_NAMES
)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Training path: /home/jadkandah/progressSoft_internship/phase-1-machine-learning-nlp/assignment/data/twitter_training.csv
Validation path: /home/jadkandah/progressSoft_internship/phase-1-machine-learning-nlp/assignment/data/twitter_validation.csv
Training file exists: True
Validation file exists: True
Training shape: (74682, 4)
Validation shape: (1000, 4)


# Testing the Preprocessing.py file

In [3]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for module_name in ["src.preprocessing", "src"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from src.preprocessing import clean_dataframe, normalize_text



In [4]:
examples = [
    "I LOVE this game!!! #Amazing",
    "Visit https://example.com now",
    "@username this update is terrible",
    "<p>Best game ever</p>",
    None,
]

for example in examples:
    print("Original:", example)
    print("Cleaned: ", normalize_text(example))
    print()

Original: I LOVE this game!!! #Amazing
Cleaned:  i love this game!!! amazing

Original: Visit https://example.com now
Cleaned:  visit now

Original: @username this update is terrible
Cleaned:  this update is terrible

Original: <p>Best game ever</p>
Cleaned:  best game ever

Original: None
Cleaned:  



In [5]:
clean_train_df = clean_dataframe(train_df)
clean_validation_df = clean_dataframe(validation_df)

print("Original training shape:", train_df.shape)
print("Clean training shape:", clean_train_df.shape)

print("Original validation shape:", validation_df.shape)
print("Clean validation shape:", clean_validation_df.shape)

Original training shape: (74682, 4)
Clean training shape: (71308, 4)
Original validation shape: (1000, 4)
Clean validation shape: (1000, 4)


In [6]:
print("Original training rows:", len(train_df))
print("Cleaned training rows:", len(clean_train_df))
print("Rows removed:", len(train_df) - len(clean_train_df))

print("\nOriginal validation rows:", len(validation_df))
print("Cleaned validation rows:", len(clean_validation_df))
print("Rows removed:", len(validation_df) - len(clean_validation_df))

Original training rows: 74682
Cleaned training rows: 71308
Rows removed: 3374

Original validation rows: 1000
Cleaned validation rows: 1000
Rows removed: 0


In [7]:
duplicate_text_rows = clean_train_df[
    clean_train_df.duplicated(subset=["text"], keep=False)
].sort_values("text")

print("Rows with duplicated text:", len(duplicate_text_rows))
print("Unique duplicated texts:", duplicate_text_rows["text"].nunique())

Rows with duplicated text: 3540
Unique duplicated texts: 881


In [8]:
text_label_counts = (
    clean_train_df
    .groupby("text")["sentiment"]
    .nunique()
)

conflicting_texts = text_label_counts[text_label_counts > 1]

print("Texts with conflicting labels:", len(conflicting_texts))

Texts with conflicting labels: 128


In [9]:
text_entity_counts = (
    clean_train_df
    .groupby("text")["entity"]
    .nunique()
)

multi_entity_texts = text_entity_counts[text_entity_counts > 1]

print(
    "Duplicate texts associated with multiple entities:",
    len(multi_entity_texts)
)

Duplicate texts associated with multiple entities: 223


In [10]:
clean_train_df = clean_train_df.drop_duplicates(
    subset=["entity", "sentiment", "text"]
).reset_index(drop=True)

In [11]:
pair_label_counts = (
    clean_train_df
    .groupby(["entity", "text"])["sentiment"]
    .nunique()
)

conflicting_pairs = pair_label_counts[pair_label_counts > 1]

print("Conflicting entity-text pairs:", len(conflicting_pairs))

Conflicting entity-text pairs: 275


In [12]:
conflicting_pair_index = set(conflicting_pairs.index)

mask = clean_train_df.apply(
    lambda row: (row["entity"], row["text"]) in conflicting_pair_index,
    axis=1
)

final_train_df = clean_train_df.loc[~mask].reset_index(drop=True)

print("Rows before conflict removal:", len(clean_train_df))
print("Rows after conflict removal:", len(final_train_df))

Rows before conflict removal: 70036
Rows after conflict removal: 69384


In [13]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

final_train_df.to_csv(
    PROCESSED_DATA_DIR / "train_clean.csv",
    index=False
)

clean_validation_df.to_csv(
    PROCESSED_DATA_DIR / "validation_clean.csv",
    index=False
)

In [14]:
from src.preprocessing import normalize_text, tokenize_text

In [15]:
examples = [
    "I absolutely LOVE this game!!!",
    "This update is soooo bad 😡",
    "@user Microsoft is doing great #Windows",
    "Not good at all...",
]

for text in examples:
    cleaned = normalize_text(text)
    tokens = tokenize_text(cleaned)

    print("Original:", text)
    print("Cleaned: ", cleaned)
    print("Tokens:  ", tokens)
    print()

Original: I absolutely LOVE this game!!!
Cleaned:  i absolutely love this game!!!
Tokens:   ['i', 'absolutely', 'love', 'this', 'game', '!', '!', '!']

Original: This update is soooo bad 😡
Cleaned:  this update is soooo bad 😡
Tokens:   ['this', 'update', 'is', 'sooo', 'bad', '😡']

Original: @user Microsoft is doing great #Windows
Cleaned:  microsoft is doing great windows
Tokens:   ['microsoft', 'is', 'doing', 'great', 'windows']

Original: Not good at all...
Cleaned:  not good at all...
Tokens:   ['not', 'good', 'at', 'all', '...']



In [16]:
token_sample = clean_train_df["text"].head(10).apply(tokenize_text)

for text, tokens in zip(
    clean_train_df["text"].head(2),
    token_sample
):
    print("Text:", text)
    print("Tokens:", tokens)
    print("\n\n")

Text: im getting on borderlands and i will murder you all ,
Tokens: ['im', 'getting', 'on', 'borderlands', 'and', 'i', 'will', 'murder', 'you', 'all', ',']



Text: i am coming to the borders and i will kill you all,
Tokens: ['i', 'am', 'coming', 'to', 'the', 'borders', 'and', 'i', 'will', 'kill', 'you', 'all', ',']





# Testing Vectorizer.py file

In [17]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for module_name in ["src.vectorizer", "src"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from src.vectorizer import build_tfidf_vectorizer



In [18]:
X_train = clean_train_df["text"]
y_train = clean_train_df["sentiment"]

X_validation = clean_validation_df["text"]
y_validation = clean_validation_df["sentiment"]

In [19]:
tfidf_vectorizer = build_tfidf_vectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_validation_tfidf = tfidf_vectorizer.transform(X_validation)

In [20]:
print("Training matrix shape:", X_train_tfidf.shape)
print("Validation matrix shape:", X_validation_tfidf.shape)
print("Matrix type:", type(X_train_tfidf))
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

Training matrix shape: (70036, 100000)
Validation matrix shape: (1000, 100000)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Vocabulary size: 100000


In [21]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("First 30 features:")
print(feature_names[:30])

First 30 features:
[' \u200d.' ' \u200d. .' ' \u200d. . .' ' \u200d. ..' ' \u200d. 🤦' '!'
 '! !' '! "' '! $' '! &' "! '" '! (' '! )' '! *' '! ,' '! -' '! .' '! . .'
 '! . . .' '! . . . .' '! . ..' '! ..' '! .. .' '! ...' '! /' '! 1' '! 10'
 '! 11' '! 11th' '! 2']


In [22]:
idf_sample = list(
    zip(
        feature_names[:2],
        tfidf_vectorizer.idf_[:2]
    )
)

idf_sample

[(' \u200d.', np.float64(7.386094328349731)),
 (' \u200d. .', np.float64(10.547341040381296))]

In [23]:
empty_train_rows = (X_train_tfidf.getnnz(axis=1) == 0).sum()
empty_validation_rows = (X_validation_tfidf.getnnz(axis=1) == 0).sum()

print("Empty training vectors:", empty_train_rows)
print("Empty validation vectors:", empty_validation_rows)

Empty training vectors: 42
Empty validation vectors: 3


In [24]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for module_name in ["src.train", "src"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from src.train import build_logistic_regression_pipeline

In [25]:
baseline_pipeline = build_logistic_regression_pipeline()

print(baseline_pipeline)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(lowercase=False, max_df=0.98,
                                 max_features=100000, min_df=2,
                                 ngram_range=(1, 2),
                                 preprocessor=<function normalize_text at 0x7ea108db85e0>,
                                 sublinear_tf=True, token_pattern=None,
                                 tokenizer=<function tokenize_text at 0x7ea096e71f80>)),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


In [26]:
sample_texts = [
    "I absolutely love this game!!!",
    "This update is horrible and broken.",
    "The product is okay, nothing special.",
]

sample_features = (
    baseline_pipeline
    .named_steps["tfidf"]
    .fit_transform(sample_texts)
)

print("Sample feature shape:", sample_features.shape)
print("Feature matrix type:", type(sample_features))

Sample feature shape: (3, 3)
Feature matrix type: <class 'scipy.sparse._csr.csr_matrix'>
